# Model 2 — random forest

This notebook trains **only Model 2** on the public UCI term-deposit subscription task. Its business analogue is STADIOEquities first-deposit activation; this data cannot model dormancy. Preprocessing and engineered features are shared with the other model for a fair comparison. See `Model2.MD`.

In [2]:
import sys
from pathlib import Path

MODEL_DIR = Path.cwd()
ROOT = MODEL_DIR.parent  # the Modelling folder
PREP_DIR = ROOT / "preprocessing and feature engineering"

assert (PREP_DIR / "data_preparation.py").is_file(), (
    f"Cannot find {PREP_DIR / 'data_preparation.py'}"
)
assert (MODEL_DIR / "model2.py").is_file(), (
    f"Cannot find {MODEL_DIR / 'model2.py'}"
)

sys.path.insert(0, str(PREP_DIR))
sys.path.insert(0, str(MODEL_DIR))

import data_preparation

# model2.py imports a module called shared.
sys.modules["shared"] = data_preparation

from data_preparation import (
    load_bank_data,
    chronological_splits,
    BankFeatureBuilder,
    train_one,
    score_one,
)
from model2 import NAME, make_classifier

DATA_PATH =  ROOT / "data" / "bank-full.csv"
OUTPUT_PATH = ROOT / "results" / "model2"

frame = load_bank_data(DATA_PATH if DATA_PATH.is_file() else None)
print(frame.shape, frame["y"].value_counts().to_dict())

(45211, 17) {'no': 39922, 'yes': 5289}


## Inspect the shared preprocessing inputs

`duration` is excluded because it is known after the phone call. The `pdays=-1` sentinel is converted into a prior-contact indicator and an optional elapsed-days value.

In [3]:
train, validation, test = chronological_splits(frame)
features = BankFeatureBuilder().fit_transform(train.drop(columns="y"))
print("Split sizes:", len(train), len(validation), len(test))
print("Model inputs:", features.columns.tolist())
print(features[["previously_contacted", "pdays_known", "total_contacts", "signed_log_balance"]].head())

Split sizes: 27126 9042 9043
Model inputs: ['age', 'balance', 'day', 'campaign', 'previous', 'pdays_known', 'previously_contacted', 'total_contacts', 'signed_log_balance', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']
   previously_contacted  pdays_known  total_contacts  signed_log_balance
0                     0          NaN               1            7.670429
1                     0          NaN               1            3.401197
2                     0          NaN               1            1.098612
3                     0          NaN               1            7.317876
4                     0          NaN               1            0.693147


## Train and evaluate Model 2

The classifier and its preprocessing pipeline fit on the earliest 60% of rows. The next 20% select an F2 threshold, and the last 20% are held out for final evaluation. Files are saved to `results/model2/`.

In [4]:
results = train_one(DATA_PATH if DATA_PATH.is_file() else None, OUTPUT_PATH, NAME, make_classifier())
print("Test metrics:", results["test"])

{
  "model": "random_forest",
  "source": "/Users/khanophidane/Documents/School/capstone/Project/capstoneProject/Modelling/data/bank-full.csv",
  "rows": 45211,
  "split_rows": {
    "train": 27126,
    "validation": 9042,
    "test": 9043
  },
  "positive_rate": {
    "train": 0.0504,
    "validation": 0.1179,
    "test": 0.3158
  },
  "excluded_post_outcome_column": "duration",
  "validation": {
    "threshold": 0.35,
    "roc_auc": 0.6137,
    "average_precision": 0.1666,
    "precision_yes": 0.1399,
    "recall_yes": 0.8171,
    "f2_yes": 0.4152,
    "confusion_matrix_no_yes": [
      [
        2622,
        5354
      ],
      [
        195,
        871
      ]
    ]
  },
  "test": {
    "threshold": 0.35,
    "roc_auc": 0.5937,
    "average_precision": 0.3823,
    "precision_yes": 0.356,
    "recall_yes": 0.7616,
    "f2_yes": 0.6203,
    "confusion_matrix_no_yes": [
      [
        2253,
        3934
      ],
      [
        681,
        2175
      ]
    ]
  }
}
Test metrics: {'

## Optional inference on new UCI-shaped rows

Put a semicolon-separated `new_customers.csv` at the project root. The target and call duration columns can be omitted. This cell scores it only when present.

In [5]:
NEW_DATA_PATH = ROOT / "new_customers.csv"
if NEW_DATA_PATH.is_file():
    score_one(NEW_DATA_PATH, OUTPUT_PATH, NAME)
else:
    print("Add new_customers.csv to run inference.")

Add new_customers.csv to run inference.
